# Day 017 Project — Solution

Demonstrates the extended tool chatbot with three tools: `calculate`, `get_weather`, and `convert_units` (a unit converter).

In [ ]:
import ollama, ast, operator

# ── Built-in tools ───────────────────────────────────────────────
def calculate(expression: str) -> str:
    allowed = {
        ast.Add: operator.add, ast.Sub: operator.sub,
        ast.Mult: operator.mul, ast.Div: operator.truediv,
        ast.Pow: operator.pow, ast.Mod: operator.mod,
        ast.USub: operator.neg,
    }
    def _eval(node):
        if isinstance(node, ast.Constant): return node.value
        if isinstance(node, ast.BinOp):
            return allowed[type(node.op)](_eval(node.left), _eval(node.right))
        if isinstance(node, ast.UnaryOp):
            return allowed[type(node.op)](_eval(node.operand))
        raise ValueError(f"Unsafe expression: {ast.dump(node)}")
    result = _eval(ast.parse(expression, mode='eval').body)
    return str(result)

def get_weather(city: str) -> str:
    temperatures = {"london": "12°C", "tokyo": "22°C", "paris": "15°C",
                    "new york": "18°C", "sydney": "24°C"}
    return f"The current temperature in {city} is {temperatures.get(city.lower(), '20°C')}."

# ── New tool: unit converter ───────────────────────────────────────────
def convert_units(value, from_unit: str, to_unit: str) -> str:
    """Convert between common units: km/miles, kg/lbs, celsius/fahrenheit."""
    value = float(value)  # model may pass as string
    conversions = {
        ("km", "miles"): lambda v: v * 0.621371,
        ("miles", "km"): lambda v: v * 1.60934,
        ("kg", "lbs"): lambda v: v * 2.20462,
        ("lbs", "kg"): lambda v: v / 2.20462,
        ("celsius", "fahrenheit"): lambda v: v * 9/5 + 32,
        ("fahrenheit", "celsius"): lambda v: (v - 32) * 5/9,
    }
    key = (from_unit.lower(), to_unit.lower())
    if key not in conversions:
        return f"Unknown conversion: {from_unit} → {to_unit}"
    result = conversions[key](value)
    return f"{value} {from_unit} = {result:.4f} {to_unit}"

In [ ]:
TOOL_REGISTRY = {
    "calculate": calculate,
    "get_weather": get_weather,
    "convert_units": convert_units,
}

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a mathematical expression. Use for any arithmetic.",
            "parameters": {
                "type": "object",
                "properties": {"expression": {"type": "string", "description": "Math expression, e.g. '12 * 34'"}},
                "required": ["expression"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Return the current temperature for a city.",
            "parameters": {
                "type": "object",
                "properties": {"city": {"type": "string", "description": "City name"}},
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "convert_units",
            "description": (
                "Convert between common units: km/miles, kg/lbs, celsius/fahrenheit. "
                "Use this when the user asks to convert a measurement."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "value": {"type": "number", "description": "The numeric value to convert"},
                    "from_unit": {"type": "string", "description": "Unit to convert from (km, miles, kg, lbs, celsius, fahrenheit)"},
                    "to_unit": {"type": "string", "description": "Unit to convert to"},
                },
                "required": ["value", "from_unit", "to_unit"],
            },
        },
    },
]

In [ ]:
def has_tool_call(response): return bool(response["message"].get("tool_calls"))
def extract_tool_call(response):
    call = response["message"]["tool_calls"][0]
    return call["function"]["name"], call["function"]["arguments"]
def execute_tool(fn_name, fn_args, registry):
    if fn_name not in registry: raise KeyError(f"Unknown tool: {fn_name!r}")
    return registry[fn_name](**fn_args)
def append_tool_result(messages, assistant_msg, tool_output):
    return messages + [assistant_msg, {"role": "tool", "content": tool_output}]
def append_turn(history, user_text, assistant_text):
    return history + [{"role": "user", "content": user_text}, {"role": "assistant", "content": assistant_text}]
def tool_turn(history, user_input, tools, registry, model="llama3.2"):
    messages = history + [{"role": "user", "content": user_input}]
    response = ollama.chat(model=model, messages=messages, tools=tools)
    if has_tool_call(response):
        fn_name, fn_args = extract_tool_call(response)
        tool_output = execute_tool(fn_name, fn_args, registry)
        messages = append_tool_result(messages, response["message"], tool_output)
        response = ollama.chat(model=model, messages=messages, tools=tools)
    reply = response["message"]["content"]
    return reply, append_turn(history, user_input, reply)
def truncate_history(history, max_turns=10):
    if not history: return []
    if history[0]["role"] == "system": system, tail = [history[0]], history[1:]
    else: system, tail = [], history
    return system + tail[-(max_turns * 2):]

## Scripted Demo

In [ ]:
SYSTEM_PROMPT = (
    "You are a helpful assistant with access to a calculator, weather lookup, and unit converter. "
    "Use the calculate tool for any arithmetic. "
    "Use the get_weather tool when asked about current temperature or weather. "
    "Use the convert_units tool when the user asks to convert between measurements."
)

history = [{"role": "system", "content": SYSTEM_PROMPT}]

turns = [
    "What is 2847 * 193?",
    "What is the weather in Tokyo?",
    "Convert 42 km to miles.",
]

print("=" * 60)
print("Day 017 Solution — Tool-Using Assistant (scripted demo)")
print("=" * 60)

for turn in turns:
    print(f"\nYou: {turn}")
    reply, history = tool_turn(history, turn, TOOLS, TOOL_REGISTRY)
    print(f"Bot: {reply}")

print("\n" + "=" * 60)
print("Demo complete.")